# Chapter 27 · Synthetic data — Lab 27

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/lab27_synthetic.ipynb)

**The factory, on one GPU.** You will run the chapter's three stations on the Lab 10 slice: **generation** — structured rephrasing (table, math, FAQ, tutorial) and two-stage Active Reading with a 1.7B instruct model, plus rejection sampling of reasoning traces with a verifier; **quality control** — a faithfulness judge against the source document, diversity by self-BLEU, the pass@k funnel; and **compliance** — the synthetic shard sent back through Lab 11's gates (13-gram decontamination, Min-K%) with the generator, prompt version and kept ratio written into its lineage. A short simulation at the end reproduces the replace-vs-accumulate collapse result.

Needs a GPU runtime (T4 is enough; ~25 minutes with the default sizes). Generation uses `transformers.generate` so it runs anywhere; a vLLM cell is provided for machines that have it.

*Omar U. Florez, PhD · Efficient Training of LLMs · Lab 27*

In [ ]:
%pip -q install datasets transformers torch accelerate pyarrow sacrebleu
import re, json, time, collections, itertools, pathlib, random
import numpy as np, matplotlib.pyplot as plt, torch
from datasets import load_dataset

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_SRC = 40 if dev == "cuda" else 6          # source documents to rephrase
LINEAGE = {"generator": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "prompt_version": "lab27-v1", "stations": {}}
random.seed(0); torch.manual_seed(0)

## 0 · Source documents and the generator

The seeds are Lab 10's cleaned slice (or a fresh FineWeb-Edu sample). The generator is SmolLM2-1.7B-Instruct — small on purpose: the chapter's evidence (BeyondWeb, FinePhrase) is that 1–3B rephrasers capture most of the gain, and a 27B one often scores lower.

In [ ]:
p = pathlib.Path("fineweb_edu_lab10/data.parquet")
if p.exists():
    import pyarrow.parquet as pq
    docs = [d for d in pq.read_table(p).to_pylist() if 150 < len(d["text"].split()) < 600][:N_SRC]
else:
    stream = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    docs = [d for d in itertools.islice(stream, 400) if 150 < len(d["text"].split()) < 600][:N_SRC]
print(len(docs), "source docs")

from transformers import AutoTokenizer, AutoModelForCausalLM
GEN = LINEAGE["generator"]
tok = AutoTokenizer.from_pretrained(GEN); tok.padding_side = "left"; tok.pad_token = tok.eos_token
gen = AutoModelForCausalLM.from_pretrained(GEN, torch_dtype=torch.bfloat16 if dev == "cuda" else torch.float32).to(dev).eval()

@torch.no_grad()
def chat(prompts, max_new_tokens=512, temperature=0.7, n=1, batch=8):
    outs = []
    for i in range(0, len(prompts), batch):
        texts = [tok.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True, tokenize=False)
                 for q in prompts[i:i+batch]]
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=3072).to(dev)
        o = gen.generate(**enc, do_sample=temperature > 0, temperature=max(temperature, 1e-5), top_p=0.95,
                         max_new_tokens=max_new_tokens, num_return_sequences=n, pad_token_id=tok.eos_token_id)
        for j, seq in enumerate(o):
            outs.append(tok.decode(seq[enc.input_ids.shape[1]:], skip_special_tokens=True))
    return outs
print("generator ready on", dev)

## 1 · What generation costs

Chapter 27.2's arithmetic, measured: tokens per second on this GPU, then the GPU-days for 1B and 1T synthetic tokens at that rate. Compare with FinePhrase's ~9,200 tok/s/GPU (1.7B model, speculative decoding, vLLM) — the gap is what a serving stack buys.

In [ ]:
t0 = time.time(); warm = chat(["Rewrite this sentence in three different ways: The model trains on data."] * 8, max_new_tokens=128)
n_tok = sum(len(tok(w).input_ids) for w in warm); dt = time.time() - t0
rate = n_tok / dt
print(f"{rate:,.0f} output tokens/s on {dev} (naive transformers.generate, batch 8)")
for T in (1e9, 1e12):
    print(f"  {T:.0e} tokens → {T / rate / 86400:,.1f} GPU-days at this rate;  {T / 9200 / 86400:,.1f} at FinePhrase's 9,200 tok/s")
LINEAGE["stations"]["throughput_tok_s"] = round(rate)

## 2 · Station 1 · Structured rephrasing

Four prompts from the chapter (the ones FinePhrase found beat plain paraphrase). Each source document becomes up to four synthetic documents; we keep the format, the source id and the prompt version on every output — that is the lineage Chapter 10 asked for.

In [ ]:
FORMATS = {
  "table":    "Rewrite the key facts of the text below as one or more Markdown tables, each with a short caption. Keep every number and name.",
  "math":     "Write 3-5 self-contained math word problems that use quantities and relations from the text below, each followed by a worked solution.",
  "faq":      "Rewrite the text below as an FAQ of 6-10 question/answer pairs that together cover all of its information.",
  "tutorial": "Rewrite the text below as a step-by-step tutorial a motivated beginner could follow.",
}
synthetic = []
t0 = time.time()
for fmt, instr in FORMATS.items():
    prompts = [f"{instr}\n\nText:\n{d['text'][:4000]}" for d in docs]
    for d, out in zip(docs, chat(prompts, max_new_tokens=512)):
        synthetic.append({"text": out, "format": fmt, "source_id": d["id"], "generator": GEN, "prompt_version": LINEAGE["prompt_version"]})
print(f"{len(synthetic)} synthetic docs in {time.time()-t0:.0f}s")
ex = next(s for s in synthetic if s["format"] == "table")
print("\n--- SOURCE ---\n", next(d for d in docs if d["id"] == ex["source_id"])["text"][:500], "\n\n--- TABLE REPHRASE ---\n", ex["text"][:700])

In [ ]:
# Optional: the same loop on vLLM (5–10× faster where it is installed)
# from vllm import LLM, SamplingParams
# vgen = LLM(GEN, dtype="bfloat16", max_model_len=4096)
# sp = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=512)
# outs = vgen.chat([[{"role": "user", "content": p}] for p in prompts], sp)
# texts = [o.outputs[0].text for o in outs]

## 3 · Station 2 · Quality: faithfulness and diversity

Two checks the chapter insists on. **Faithfulness**: the source document is the verifier — a judge prompt scores each rewrite 1–5 for staying true to it (same small model as judge; in production, calibrate it with Lab 11's human sample). **Diversity**: self-BLEU across the outputs of one format — low is good; "template collapse" (FinePhrase's term for a generator that follows the format too rigidly) shows up as high self-BLEU.

In [ ]:
JUDGE = ("You are checking a rewrite against its source. Score 1-5 how faithful the rewrite is to the source: "
         "5 = every fact matches, no invented facts; 1 = mostly invented or contradicts the source. Answer with the digit only.\n\n"
         "SOURCE:\n{src}\n\nREWRITE:\n{rw}\n\nScore:")
src_by_id = {d["id"]: d["text"] for d in docs}
verdicts = chat([JUDGE.format(src=src_by_id[s["source_id"]][:2500], rw=s["text"][:1500]) for s in synthetic], max_new_tokens=4, temperature=0)
for s, v in zip(synthetic, verdicts):
    m = re.search(r"[1-5]", v); s["faithfulness"] = int(m.group()) if m else None
by_fmt = collections.defaultdict(list)
for s in synthetic:
    if s["faithfulness"]: by_fmt[s["format"]].append(s["faithfulness"])
print({k: round(np.mean(v), 2) for k, v in by_fmt.items()})
kept = [s for s in synthetic if (s["faithfulness"] or 0) >= 4]
print(f"kept {len(kept)} / {len(synthetic)} at faithfulness ≥ 4   (kept : generated ≈ 1 : {len(synthetic)/max(1,len(kept)):.1f})")
LINEAGE["stations"]["rephrase"] = {"generated": len(synthetic), "kept": len(kept), "faithfulness_threshold": 4,
                                   "mean_faithfulness_by_format": {k: float(np.mean(v)) for k, v in by_fmt.items()}}

In [ ]:
import sacrebleu
def self_bleu(texts, k=12):
    texts = texts[:k]; scores = []
    for i, t in enumerate(texts):
        others = [x for j, x in enumerate(texts) if j != i]
        if others: scores.append(sacrebleu.sentence_bleu(t, others).score)
    return float(np.mean(scores)) if scores else float("nan")
sb = {fmt: self_bleu([s["text"] for s in synthetic if s["format"] == fmt]) for fmt in FORMATS}
sb["source docs"] = self_bleu([d["text"] for d in docs])
plt.figure(figsize=(6,3)); plt.bar(list(sb), list(sb.values()), color=[BLUE]*4 + [GRAY])
plt.ylabel("self-BLEU (lower = more diverse)"); plt.title("diversity by format; watch for template collapse")
print({k: round(v, 1) for k, v in sb.items()})
LINEAGE["stations"]["self_bleu"] = sb

> **Exercise 3.1.** Set `temperature=0.2` in the rephrasing loop and recompute self-BLEU. Then try a rigid prompt ("Use exactly this template: …"). Both should raise self-BLEU — that is the collapse the chapter warns about, visible before any training.

## 4 · Active Reading, two stages

Chapter 27.4: the model writes document-specific learning strategies, then applies each one independently. Diversity comes from the strategies, not from sampling temperature. We run it on a handful of documents and look at what the model decided to do with each.

In [ ]:
STRAT = ("You are studying this document to learn and remember everything in it. Write learning strategies "
         "SPECIFIC to this document, one per line, each starting with '## '. Write 4 to 6 strategies.\n\nDocument:\n{doc}")
APPLY = "Here's a learning strategy: {s}\nApply this strategy to the following document.\n\nDocument:\n{doc}"
N_AR = 5 if dev == "cuda" else 2
ar_docs = docs[:N_AR]
plans = chat([STRAT.format(doc=d["text"][:3500]) for d in ar_docs], max_new_tokens=200, temperature=0.7)
strategies = [[l[3:].strip() for l in p.splitlines() if l.startswith("## ")][:6] for p in plans]
print("strategies for doc 0:"); [print("  ·", s) for s in strategies[0]]
apply_prompts, meta = [], []
for d, ss in zip(ar_docs, strategies):
    for s in ss: apply_prompts.append(APPLY.format(s=s, doc=d["text"][:3500])); meta.append((d["id"], s))
aug = chat(apply_prompts, max_new_tokens=400, temperature=0.7)
active = [{"text": a, "strategy": s, "source_id": sid, "format": "active_reading", "generator": GEN,
           "prompt_version": LINEAGE["prompt_version"]} for a, (sid, s) in zip(aug, meta)]
print(f"\n{len(active)} augmentations from {len(ar_docs)} docs ({len(active)/len(ar_docs):.1f} per doc)")
print("\n--- one augmentation ---\nstrategy:", active[0]["strategy"], "\n", active[0]["text"][:600])
print("\nself-BLEU, active reading outputs for doc 0:", round(self_bleu([a["text"] for a in active if a["source_id"] == ar_docs[0]["id"]]), 1),
      " vs. four format rewrites of the same doc:", round(self_bleu([s["text"] for s in synthetic if s["source_id"] == ar_docs[0]["id"]]), 1))
LINEAGE["stations"]["active_reading"] = {"docs": len(ar_docs), "augmentations": len(active)}

> **Exercise 4.1.** Run the faithfulness judge of §3 over the Active Reading outputs and group by strategy name. The chapter's rule: a strategy whose outputs hallucinate is dropped *by name*, not output by output.

## 5 · Rejection sampling: the pass@k funnel

Chapter 27.6 on GSM8K train: sample $k$ traces per prompt, verify the final answer, keep what passes, drop prompts with all-fail (too hard) or all-pass (too easy). Then the arithmetic: $\text{pass@}k = 1-(1-p)^k$ and expected kept traces $kp$.

In [ ]:
gsm = list(itertools.islice(load_dataset("openai/gsm8k", "main", split="train", streaming=True), 24 if dev == "cuda" else 4))
def gold(a): return a.split("####")[-1].strip().replace(",", "")
def last_number(s):
    m = re.findall(r"-?\d+(?:\.\d+)?", s.replace(",", "")); return m[-1] if m else None
K = 6
prompts = [ex["question"] + "\nSolve step by step and end with 'The answer is <number>'." for ex in gsm]
traces = chat(prompts, max_new_tokens=320, temperature=1.0, n=K, batch=2)
kept_traces, funnel = [], collections.Counter()
for i, ex in enumerate(gsm):
    ts = traces[i*K:(i+1)*K]; ok = [t for t in ts if last_number(t) == gold(ex["answer"])]
    funnel["generated"] += K; funnel["verified"] += len(ok)
    if 0 < len(ok) < K:
        keep = sorted(set(ok), key=len)[:3]; funnel["kept"] += len(keep)
        kept_traces += [{"prompt": ex["question"], "trace": t, "pass_rate": len(ok)/K} for t in keep]
    elif len(ok) == 0: funnel["too_hard_prompts"] += 1
    else: funnel["too_easy_prompts"] += 1
print(dict(funnel)); print(f"kept : generated = 1 : {funnel['generated']/max(1,funnel['kept']):.1f}")
k = np.array([1, 2, 4, 8, 16, 32, 64])
plt.figure(figsize=(6,3))
for pp in (0.3, 0.1, 0.02): plt.plot(k, 1-(1-pp)**k, marker="o", label=f"p = {pp}")
plt.xscale("log", base=2); plt.xlabel("samples k"); plt.ylabel("pass@k"); plt.legend(); plt.title("where extra samples buy traces (Figure 27.5)")
LINEAGE["stations"]["rejection_sampling"] = dict(funnel) | {"k": K}

## 6 · Station 3 · Compliance: back through Lab 11's gates

The generator has seen the benchmarks, so the synthetic shard is the likeliest place for a leak. Decontaminate it with 13-grams against GSM8K/MMLU **before** it is mixed in, run Min-K% under the base model on a sample, and write the lineage record into the shard.

In [ ]:
def ngrams(text, n=13):
    w = text.lower().split(); return {" ".join(w[i:i+n]) for i in range(len(w)-n+1)}
trip = set()
for it in load_dataset("openai/gsm8k", "main", split="test"): trip |= ngrams(it["question"] + " " + it["answer"])
for it in itertools.islice(load_dataset("cais/mmlu", "all", split="test", streaming=True), 3000): trip |= ngrams(it["question"] + " " + " ".join(it["choices"]))
shard = kept + active
clean = [s for s in shard if not (ngrams(s["text"]) & trip)]
print(f"decontamination: {len(shard) - len(clean)} removed of {len(shard)}")

# Min-K% under a base model (the student that will train on this), sample
base_name = "HuggingFaceTB/SmolLM2-360M"
btok = AutoTokenizer.from_pretrained(base_name); base = AutoModelForCausalLM.from_pretrained(base_name).to(dev).eval()
@torch.no_grad()
def min_k(text, k=0.2):
    ids = btok(text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(dev)
    lp = torch.log_softmax(base(ids).logits[0, :-1].float(), -1).gather(1, ids[0, 1:, None])[:, 0].cpu().numpy()
    return float(np.sort(lp)[: max(1, int(k*len(lp)))].mean())
mk_syn = [min_k(s["text"]) for s in clean[:40]]; mk_src = [min_k(d["text"]) for d in docs[:20]]
print(f"Min-K% mean  synthetic {np.mean(mk_syn):.2f}   source {np.mean(mk_src):.2f}   (closer to 0 = less surprising to the base)")
LINEAGE["stations"]["compliance"] = {"decontam_removed": len(shard) - len(clean), "min_k_synthetic": float(np.mean(mk_syn)), "min_k_source": float(np.mean(mk_src))}

import pyarrow as pa, pyarrow.parquet as pq
out = pathlib.Path("synthetic_lab27"); out.mkdir(exist_ok=True)
cols = ["text", "format", "source_id", "generator", "prompt_version"]
pq.write_table(pa.Table.from_pylist([{c: s.get(c) for c in cols} for s in clean]), out / "shard.parquet")
LINEAGE["output"] = {"n_docs": len(clean), "file": "shard.parquet", "mix_rule": "train at synthetic:real ≈ 1:1; real data never leaves the mix"}
(out / "README.md").write_text("# synthetic_lab27\n\n```json\n" + json.dumps(LINEAGE, indent=1) + "\n```\n")
print(json.dumps(LINEAGE, indent=1)[:1500])

## 7 · Collapse needs replacement: a two-line simulation

Chapter 27.7's Gaussian recursion. Each generation fits a Gaussian to $M$ samples; under **replacement** it samples from the previous fit only, under **accumulation** from all data so far. Variance collapses geometrically in one case and stays bounded in the other.

In [ ]:
rng = np.random.default_rng(0); M, G = 200, 40
real = rng.normal(0, 1, M)
var_replace, var_accum = [], []
cur = real.copy(); pool = real.copy()
for g in range(G):
    cur = rng.normal(cur.mean(), cur.std(), M);            var_replace.append(cur.var())      # replace: last generation only
    synth = rng.normal(pool.mean(), pool.std(), M); pool = np.concatenate([pool, synth]); var_accum.append(pool.var())  # accumulate
plt.figure(figsize=(6,3)); plt.plot(var_replace, color=AMBER, lw=2, label="replace"); plt.plot(var_accum, color=NAVY, lw=2, label="accumulate (real + all synthetic)")
plt.plot([1*(1-1/M)**g for g in range(G)], color=AMBER, ls="--", lw=1, label="σ²(1−1/M)^n")
plt.xlabel("generation"); plt.ylabel("variance of the data"); plt.legend(); plt.title("Figure 27.6, simulated")
print(f"after {G} generations: replace {var_replace[-1]:.3f}  accumulate {var_accum[-1]:.3f}  (started at {real.var():.3f})")

> **Exercise 7.1.** Add the quality station to the simulation: before accumulating, drop synthetic samples more than 2σ from the real mean (a "verifier"). Does the accumulate curve change? Then replace the verifier with one that keeps only samples *within* 0.5σ — a judge with a strong preference — and watch the variance fall even under accumulation. That is the quiet collapse: a biased filter, not recursion.

## What you reproduced

* the cost of generation on your own GPU, against the 9,200 tok/s production figure;
* four structured rephrasing formats with lineage on every output, a faithfulness judge with a 1 : N kept ratio, and self-BLEU as the template-collapse detector;
* two-stage Active Reading and its diversity advantage over fixed rewrites of the same document;
* the rejection-sampling funnel with pass@k arithmetic on GSM8K;
* the compliance station: decontaminate before mixing, Min-K% against the base, lineage in the shard;
* the replace-vs-accumulate collapse result.

The shard in `synthetic_lab27/` is ready to be mixed 1:1 with the Lab 10 slice for the Phase 2 capstone pilot.